# DAG-MCTS Workflow with Real LLM Agent

This notebook demonstrates the full DAG-MCTS algorithm using the `DAGMCTSAgent` class with a real LLM (not mock).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import scanpy as sc
import numpy as np
from pathlib import Path
import json

# Load pbmc3k data
adata_path = Path("../data/pbmc3k.h5ad")
if not adata_path.exists():
    print("Downloading pbmc3k dataset...")
    adata = sc.datasets.pbmc3k()
    adata.write(adata_path)
else:
    adata = sc.read(adata_path)

# Preprocessing
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.pca(adata, n_comps=20)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

print(f"Loaded {adata.n_obs} cells, {adata.n_vars} genes")
print(f"Clusters: {adata.obs['leiden'].nunique()}")

Loaded 2700 cells, 32738 genes
Clusters: 9


In [2]:
import sklearn.metrics as skm

# Compute cluster statistics
X = adata.obsm["X_pca"]
labels = adata.obs["leiden"].astype(str).values
sil_per_cell = skm.silhouette_samples(X, labels)
adata.obs["silhouette"] = sil_per_cell

sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", n_genes=10)

cluster_stats = {}
for c in sorted(set(labels)):
    mask = labels == c
    markers = adata.uns["rank_genes_groups"]["names"][c][:10].tolist()
    stats = {
        "n": int(mask.sum()),
        "silhouette": float(sil_per_cell[mask].mean()),
        "het": 1.0 - float(sil_per_cell[mask].mean()),
        "conf": float(max(sil_per_cell[mask].mean(), 0.1)),
        "markers": markers,
    }
    cluster_stats[c] = stats

# Select cluster with highest heterogeneity
target_c = max(cluster_stats, key=lambda c: cluster_stats[c]["het"])
target_markers = cluster_stats[target_c]["markers"]

print(f"Target cluster: {target_c}")
print(f"Cluster size: {cluster_stats[target_c]['n']} cells")
print(f"Heterogeneity: {cluster_stats[target_c]['het']:.3f}")
print(f"Top markers: {target_markers[:5]}")

Target cluster: 3
Cluster size: 396 cells
Heterogeneity: 0.967
Top markers: ['CCL5', 'NKG7', 'IL32', 'B2M', 'GZMA']


## Load API Key and LLM Config

In [3]:
# Load API key from file
api_key_path = Path("../api_key")
if api_key_path.exists():
    api_key = api_key_path.read_text().strip()
    print(f"API key loaded: {api_key[:10]}...")
else:
    raise FileNotFoundError("api_key file not found in project root")

# Load LLM config
import yaml
config_path = Path("../src/cellhashtag/config/config.yaml")
config = yaml.safe_load(config_path.read_text())
llm_config = config["llm"]

print(f"LLM provider: {llm_config['provider']}")
print(f"LLM model: {llm_config['model']}")
print(f"API base: {llm_config['api_base']}")

API key loaded: sk-sp-D.HD...
LLM provider: anthropic
LLM model: qwen3.6-plus
API base: https://token-plan.cn-beijing.maas.aliyuncs.com/apps/anthropic


## Initialize DAG-MCTS Agent

In [4]:
from cellhashtag.agents import DAGMCTSAgent
import os

# Set environment variables for LLM
os.environ["ANTHROPIC_API_KEY"] = api_key
os.environ["ANTHROPIC_BASE_URL"] = llm_config["api_base"]

# Create agent with real LLM
agent = DAGMCTSAgent(
    model=llm_config["model"],
    model_provider=llm_config["provider"],
    enable_tree_viz=True,
    output_dir="../output",
)

print(f"DAGMCTSAgent initialized with model: {llm_config['model']}")

DAGMCTSAgent initialized with model: qwen3.6-plus


## Run DAG-MCTS Algorithm

In [5]:
# Run full DAG-MCTS with real LLM
result = agent.run(
    cluster_id=target_c,
    markers=target_markers,
    tissue="pbmc",
    adata=adata,
    max_iterations=8,
    cpuct=1.5,
    tau0=1.0,
    alpha=0.3,
    early_stop_threshold=2.0,
    confidence_threshold=0.7,
)

print(f"DAG nodes created  = {result['tree_stats']['total_nodes']}")
print(f"best cell type     = {result['cell_type']}")
print(f"confidence         = {result['confidence']:.3f}")
print(f"reasoning          = {result['reasoning'][:100]}")

adata.X seems to be already log-transformed.


DAG nodes created  = 7
best cell type     = cell_type=CD8+ T Cell
confidence         = 2.178
reasoning          = 


## Visualize Search Tree

In [6]:
from pathlib import Path as _P
tree_dir = _P("../output/trees") / target_c
html_files = sorted(tree_dir.glob("iter_*.html")) if tree_dir.exists() else []
print(f"{len(html_files)} tree viz files in {tree_dir}")
for f in html_files:
    print(f"  - {f.name}")

3 tree viz files in ../output/trees/3
  - iter_000.html
  - iter_001.html
  - iter_9999.html


In [7]:
from IPython.display import IFrame, display
if html_files:
    last = html_files[-1]
    print(f"Embedding {last.name}:")
    display(IFrame(src=str(last.resolve()), width="100%", height="600"))
else:
    print("No tree viz files found")

Embedding iter_9999.html:


## Final Annotation Result

In [8]:
# Display final annotation
print("=" * 60)
print("FINAL ANNOTATION")
print("=" * 60)
print(f"Cluster:           {result['cluster']}")
print(f"Cell type:         {result['cell_type']}")
print(f"Confidence:        {result['confidence']:.3f}")
print(f"Reasoning:         {result['reasoning']}")
print()
if result['evidence']:
    print("Evidence:")
    for ev in result['evidence']:
        print(f"  - Source: {ev['source']}")
        print(f"    Node: {ev['node_id']}")
        print(f"    Q-value: {ev['q_value']:.3f}")
        print(f"    Visits: {ev['visits']}")
        print(f"    Depth: {ev['depth']}")
        print(f"    Hallucination risk: {ev['hallucination_risk']}")
print()
print("Tree stats:")
for k, v in result['tree_stats'].items():
    print(f"  {k}: {v}")

FINAL ANNOTATION
Cluster:           3
Cell type:         cell_type=CD8+ T Cell
Confidence:        2.178
Reasoning:         

Evidence:
  - Source: dag_mcts
    Node: 3_root->AssignLabel:cell_type=CD8+ T Cell
    Q-value: 2.178
    Visits: 1
    Depth: 1
    Hallucination risk: unknown

Tree stats:
  total_nodes: 7
  max_depth: 2
  dag_nodes_with_multiple_parents: 0


## Compare with Naive Marker Vote

In [9]:
# Compare with simple marker voting
KNOWN_PANELS = {
    "CD3D":   ("T cell",        ["CD3D", "CD3E", "CD4", "CD8A"]),
    "CD79A":  ("B cell",        ["CD79A", "MS4A1", "CD19"]),
    "CD14":   ("Monocyte",      ["CD14", "LYZ", "S100A8", "S100A9"]),
    "NKG7":   ("NK cell",       ["NKG7", "GNLY", "KLRD1"]),
    "FCGR3A": ("FCGR3A+ Mono",  ["FCGR3A", "MS4A7", "LYZ"]),
    "PPBP":   ("Megakaryocyte", ["PPBP", "PF4", "GP1BB"]),
}

def naive_marker_vote(markers, panels=KNOWN_PANELS):
    scores = {}
    for ct, panel in panels.values():
        scores[ct] = len(set(panel) & set(markers)) / len(panel)
    return max(scores, key=scores.get), scores

naive_type, naive_scores = naive_marker_vote(target_markers)
print(f"Naive marker vote: {naive_type}")
print(f"Scores: {naive_scores}")
print()
print(f"DAG-MCTS result:   {result['cell_type']} (confidence {result['confidence']:.3f})")
print(f"Match: {naive_type == result['cell_type']}")

Naive marker vote: NK cell
Scores: {'T cell': 0.25, 'B cell': 0.0, 'Monocyte': 0.0, 'NK cell': 0.3333333333333333, 'FCGR3A+ Mono': 0.0, 'Megakaryocyte': 0.0}

DAG-MCTS result:   cell_type=CD8+ T Cell (confidence 2.178)
Match: False
